# Continue original ApET: MMStar pilot

Copy these three cells into the **existing live smoke notebook** after pushing the update. Keep Internet and T4 ×2 enabled. The locked environment and cached weights are reused. Run the cells in order; after a pilot failure, the last cell can still bundle saved evidence. Full instructions: `research/ORIGINAL_MMSTAR_PILOT.md`.

## 1. Update and sync (adds only PyArrow)

In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

repo = Path("/kaggle/working/CSE590-ApET")
original_python = repo / "environments/original/.venv/bin/python"
subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)
subprocess.run(
    ["bash", "bootstrap/setup_original_linux.sh", "--require-cuda",
     "--expected-gpus", "2", "--output", "/kaggle/working/original-runtime-mmstar.json"],
    cwd=repo, check=True,
)


## 2. Evaluate 24 images across four settings

One model load; 96 scored predictions plus warmups and a baseline repeat. These are diagnostic scores; review raw parsing failures before scaling.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
pilot_report = Path(f"/kaggle/working/original-mmstar-24-{stamp}.json")
subprocess.run(
    [str(original_python), "-I", "scripts/eval_original_mmstar.py",
     "--per-category", "4", "--configurations",
     "baseline", "input-only", "decoder-only", "combined",
     "--output", str(pilot_report)],
    cwd=repo, check=True,
)


## 3. Inspect and download evidence

Download the ZIP before ending the session. It contains reports, predictions, images, and environment files; weights are excluded.

In [ ]:
import zipfile
from IPython.display import FileLink, display

result = json.loads(pilot_report.read_text())
print(json.dumps({
    key: result.get(key) for key in
    ["status", "error", "progress", "summaries", "comparisons", "baseline_state_check"]
}, indent=2))

bundle = pilot_report.with_suffix(".zip")
image_dir = pilot_report.parent / (pilot_report.stem + "-images")
files = [pilot_report, pilot_report.with_suffix(".jsonl")]
if image_dir.is_dir():
    files.extend(sorted(path for path in image_dir.rglob("*") if path.is_file()))
files.extend(pilot_report.parent / name for name in [
    "original-runtime-mmstar.json", "original-runtime-smoke.json",
    "original-apet-preflight.json", "original-apet-smoke.json", "original-apet-smoke.png",
])
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        if path.is_file():
            archive.write(path, str(path.relative_to(pilot_report.parent)))
    for name in ["pyproject.toml", "uv.lock", ".python-version"]:
        path = repo / "environments/original" / name
        archive.write(path, f"environment/{name}")
print(f"Download: {bundle}")
display(FileLink(str(bundle)))
